<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Sensor_fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install nuscenes-devkit open3d opencv-python matplotlib ultralytics

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud, RadarPointCloud
import cv2
import numpy as np

DATA_ROOT = "/content/drive/MyDrive/nuscenes"

nusc = NuScenes(
    version="v1.0-mini",
    dataroot=DATA_ROOT ,
    verbose=True
)

sample = nusc.sample[0]

cam_token   = sample["data"]["CAM_FRONT"]
lidar_token = sample["data"]["LIDAR_TOP"]
radar_token = sample["data"]["RADAR_FRONT"]

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 11.908 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


Camera, LiDAR, Radar code

In [5]:
# Camera

cam_sd = nusc.get("sample_data", cam_token)
img = cv2.imread(nusc.get_sample_data_path(cam_token))

In [6]:
# LiDAR

lidar_sd = nusc.get("sample_data", lidar_token)
lidar_pc = LidarPointCloud.from_file(
    nusc.get_sample_data_path(lidar_token)
)
points_lidar = lidar_pc.points[:3].T  # (N,3)

In [ ]:
# Radar

radar_pc = RadarPointCloud.from_file(
    nusc.get_sample_data_path(radar_token)
)

radar_pts = radar_pc.points
radar_xyv = radar_pts[[0,1,8]].T  # x, y, velocity

Coordinate transformations

In [7]:
def get_sensor_to_ego(nusc, sd_token):
    sd = nusc.get("sample_data", sd_token)
    cs = nusc.get("calibrated_sensor", sd["calibrated_sensor_token"])
    return cs

In [15]:
from nuscenes.utils.geometry_utils import view_points
from pyquaternion import Quaternion

def project_lidar_to_image(nusc, lidar_token, cam_token, points):
    cam_sd = nusc.get("sample_data", cam_token)
    lidar_sd = nusc.get("sample_data", lidar_token)

    cam_cs = nusc.get("calibrated_sensor", cam_sd["calibrated_sensor_token"])
    lidar_cs = nusc.get("calibrated_sensor", lidar_sd["calibrated_sensor_token"])

    K = np.array(cam_cs["camera_intrinsic"])

    # Convert quaternions to rotation matrices
    lidar_rot_matrix = Quaternion(lidar_cs["rotation"]).rotation_matrix
    cam_rot_matrix = Quaternion(cam_cs["rotation"]).rotation_matrix

    # Transform lidar → ego → cam
    lidar_pc = LidarPointCloud(points.T)
    lidar_pc.rotate(lidar_rot_matrix)
    lidar_pc.translate(lidar_cs["translation"])

    lidar_pc.rotate(np.linalg.inv(cam_rot_matrix))
    lidar_pc.translate(-np.array(cam_cs["translation"]))

    pts_img = view_points(lidar_pc.points[:3], K, normalize=True)
    return pts_img[:2].T

Camera detection

In [16]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model(img)[0]

detections = []
for box in results.boxes:
    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
    detections.append((x1, y1, x2, y2))


0: 384x640 1 person, 4 cars, 1 truck, 93.6ms
Speed: 3.3ms preprocess, 93.6ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)


LiDAR and Camera fusion

In [17]:
import numpy as np

# points_lidar currently has shape (N, 3)
# The project_lidar_to_image function (specifically the LidarPointCloud constructor inside it)
# expects a (4, N) array for homogeneous coordinates.
# So, we need to transform points_lidar from (N, 3) to (N, 4) by adding a column of ones.
points_lidar_homogeneous = np.concatenate((points_lidar, np.ones((points_lidar.shape[0], 1))), axis=1)

img_pts = project_lidar_to_image(
    nusc, lidar_token, cam_token, points_lidar_homogeneous
)

fused = []

for (x1, y1, x2, y2) in detections:
    mask = (
        (img_pts[:,0] > x1) &
        (img_pts[:,0] < x2) &
        (img_pts[:,1] > y1) &
        (img_pts[:,1] < y2)
    )

    cluster = points_lidar[mask]

    if len(cluster) > 10:
        centroid = cluster.mean(axis=0)
        fused.append({"pos": centroid})

Radar

In [18]:
def attach_velocity(objects, radar_xyv):
    for obj in objects:
        dx = radar_xyv[:,0] - obj["pos"][0]
        dy = radar_xyv[:,1] - obj["pos"][1]
        d = np.sqrt(dx**2 + dy**2)
        idx = np.argmin(d)
        obj["velocity"] = radar_xyv[idx, 2]

In [19]:
for o in fused:
    print({
        "position_xyz": o["pos"].tolist(),
        "velocity_mps": o.get("velocity", None)
    })

{'position_xyz': [2.8114967346191406, -1.9281402826309204, -1.282307744026184], 'velocity_mps': None}
{'position_xyz': [1.588674545288086, -0.7648340463638306, -1.853225588798523], 'velocity_mps': None}
{'position_xyz': [1.8548603057861328, -1.9570156335830688, -1.8327701091766357], 'velocity_mps': None}
{'position_xyz': [1.5610787868499756, -0.31458035111427307, -1.8113596439361572], 'velocity_mps': None}
{'position_xyz': [-2.7997682094573975, -9.54149055480957, -2.0446717739105225], 'velocity_mps': None}
{'position_xyz': [2.371514081954956, -4.002439022064209, -1.6518049240112305], 'velocity_mps': None}
